# L12 · GRPO and Verifiable Rewards

## Goal

- compute group-relative advantages
- explain the critic-free trade-off
- diagnose zero variance and length bias

## Setup

This cell fixes CPU, seed, offline status, and the split hash first. Toy code uses deterministic CPU operations; package trainers retain their strict global default.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L12:toy:42").hexdigest()
print(f"lesson=L12 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L12 language=en profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.10.12 rl_study=0.1.0.dev0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:c261f138b02c7389fc58467b8b2b0502ad850b154b2d0a4ddd24631066a0431d data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. Position and core equation

⏱ 5 min · 1/3 section · [CORE]

Position: LLM policy + verifier → **GRPO and RLVR** → DAPO

$$\hat A_i=\frac{r_i-\operatorname{mean}(r_{1:G})}{\operatorname{std}(r_{1:G})+\epsilon}$$

GRPO samples G completions for one prompt and uses group-mean reward as a baseline, removing a separate critic. RLVR uses verifiable rewards such as correctness and format, reducing reward-model ambiguity.

### 2. Run with small numbers

⏱ 6 min · 2/3 section · [CORE]

**Predict first:** If every reward in a group is 2, are normalized advantages NaN or zero? Write an answer for 20 seconds, then run the cell.

<details><summary>Show answer</summary>A safe implementation returns zeros and marks the group uninformative.</details>

In [2]:
from rl_study.algorithms.grpo import group_relative_advantages, rloo_advantages
group_rewards = torch.tensor([[0.0, 0.0, 1.0, 1.0], [2.0, 2.0, 2.0, 2.0]])
grpo_adv = group_relative_advantages(group_rewards)
rloo_adv = rloo_advantages(group_rewards)
print({"grpo_advantages": grpo_adv.advantages.tolist(),
       "informative": grpo_adv.informative_groups.tolist(),
       "rloo_first_group": rloo_adv[0].tolist(),
       "all_finite": bool(torch.isfinite(grpo_adv.advantages).all())})

{'grpo_advantages': [[-0.9998000264167786, -0.9998000264167786, 0.9998000264167786, 0.9998000264167786], [0.0, 0.0, 0.0, 0.0]], 'informative': [True, False], 'rloo_first_group': [-0.6666666865348816, -0.6666666865348816, 0.6666666269302368, 0.6666666269302368], 'all_finite': True}


### 3. Implementation anatomy

⏱ 6 min · 3/3 section · [DEEP DIVE]

**Why this implementation:** Adding epsilon alone keeps values finite but can amplify nearly identical rewards. An explicit zero-variance mask exposes wasted sampling budget. RLOO is an alternative baseline.

**Common trap:** Normalizing rewards across different prompts mixes prompt difficulty into credit. Preserve the group axis and prompt IDs. Regression tests: `test_group_relative_advantages_and_zero_variance_group`.

**Checkpoint:** Continue when you can explain just one printed value.

## Checks

In [3]:
assert grpo_adv.informative_groups.tolist() == [True, False]
assert torch.equal(grpo_adv.advantages[1], torch.zeros(4))
print("checks=passed")

checks=passed


**Recall:** What cost shrinks when removing the critic, and what sampling dependency grows? Answer in one or two sentences.

## Mistakes I Revisit

- Assuming a finite loss proves the implementation is correct.
- Merging `terminated` with `truncated`, or prompt with action.
- Turning one tiny seed into an algorithm ranking.

## 60-Second Recap

- **Run conclusion:** The first group produced advantages near ±1; the constant-reward second group produced exact zeros with `informative=False`.
- Executable checks: `test_group_relative_advantages_and_zero_variance_group`.
- The output is a fixed-seed toy run, not a paper-scale result.

## Next Steps

1. L13 decomposes zero-variance groups, length bias, and clipping through DAPO, Dr.GRPO, and GSPO variants.
2. Break one `[CORE]` assertion and read the failure.
3. Open the package test and connect the notebook equation to its production guard.

## Sources

- `deepseekmath-grpo-2024` — `docs/sources.yml`
- `rloo-2024` — `docs/sources.yml`
- `repo-deepseek-math` — `docs/sources.yml`